In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "fact_reviews")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_reviews_s = spark.read.format("delta").load(f"{silver_base}olist_order_reviews_dataset")
df_orders_s = spark.read.format("delta").load(f"{silver_base}olist_orders_dataset")

# Gold Dimension Sources
df_dim_orders = spark.read.format("delta").load(f"{gold_base}dim_orders")
df_dim_customers = spark.read.format("delta").load(f"{gold_base}dim_customers")

# Aggregation of reviews
df_agg_reviews = (df_reviews_s
                  .groupBy("order_id")
                  .agg(
                      F.avg(F.col("review_score")).cast("decimal(3,2)").alias("avg_review_score"),
                      F.max("review_creation_date").alias("latest_review_date"),
                      F.max("review_answer_timestamp").alias("latest_answer_timestamp")
                  ))

# Join
df_enriched = (df_agg_reviews.alias("ar")
               .join(df_orders_s.alias("o"), F.col("ar.order_id") == F.col("o.order_id"), "left")
               .join(df_dim_orders.alias("do"), F.col("ar.order_id") == F.col("do.order_id"), "left")
               .join(df_dim_customers.alias("dc"), 
                     (F.col("o.customer_id") == F.col("dc.customer_id")) &
                     (F.col("o.order_purchase_timestamp") >= F.col("dc.start_date")) &
                     ((F.col("o.order_purchase_timestamp") < F.col("dc.end_date")) | (F.col("dc.end_date").isNull())), 
                     "left"))

df_fact_reviews = (df_enriched.select(
                    F.md5(F.col("ar.order_id")).cast("string").alias("review_key"),
                    F.coalesce(F.col("do.order_key"), F.lit("-1")).cast("string").alias("order_key"),
                    F.coalesce(F.col("dc.customer_key"), F.lit("-1")).cast("string").alias("customer_key"),
                    "ar.avg_review_score",
                    "ar.latest_review_date",
                    "ar.latest_answer_timestamp",
                    F.datediff(F.col("ar.latest_answer_timestamp"), F.col("ar.latest_review_date")).alias("review_response_lag_days"),
                    F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
))

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    # Merge Condition
    data_cols = ["avg_review_score", "latest_review_date", "latest_answer_timestamp"]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    (dt_gold.alias("target")
     .merge(df_fact_reviews.alias("source"), "target.order_key = source.order_key")
     .whenMatchedUpdateAll(condition = change_condition)
     .whenNotMatchedInsertAll()
     .execute())
    print(f"--> [Merge] {target_table} updated.")

else:
    #First time load
    df_fact_reviews.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

    # ZORDERING on order id
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (order_key)")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")